# DamiCore analyzes DamiCore

DAMICORE was introduced as **Data-Mining of Code Repositories**: raw code can be treated as a collection of objects, compared with Normalized Compression Distance (NCD), arranged by Neighbor Joining, and partitioned into communities. This notebook applies that idea to the current implementation itself.

**Question:** what similarity structure does DamiCore recover from its own production Python modules?

This is an exploratory structural view, not an architecture-quality score. A shared cluster means that the chosen representation contains compressible similarity; it does not imply that modules should be merged or that package boundaries are wrong.

## Experimental contract

One object is one tracked Python **implementation module** from the five public DamiCore distributions.

- `packages/*/src/**/*.py` is considered only for the public DamiCore packages.
- `__init__.py` is excluded because it mainly declares package surfaces rather than implementation behavior.
- Tests, notebooks, documentation, generated files, and the private `synthetic_data` package are excluded.
- Files are analyzed as raw bytes. There is no tokenization, AST normalization, or language-specific feature extraction.
- The notebook uses the exact public pipeline: NCD → deterministic Neighbor Joining → FastGreedy clustering.
- A single worker keeps notebook execution simple and avoids process-pool behavior; it does not change the semantic result.

Run this notebook from a repository checkout after `make install`. It needs no network access.

In [ ]:
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from damicore import ExecutionConfig, estimate, run
from IPython.display import display

PUBLIC_PACKAGES = {
    "damicore",
    "damicore_normalizer",
    "damicore_distance",
    "damicore_tree_builder",
    "damicore_clusterizer",
}

repo_root = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True,
    ).strip()
)

tracked = subprocess.check_output(
    ["git", "ls-files", "packages"],
    cwd=repo_root,
    text=True,
).splitlines()

relative_modules = []
for relative in tracked:
    path = Path(relative)
    if (
        len(path.parts) >= 5
        and path.parts[0] == "packages"
        and path.parts[1] in PUBLIC_PACKAGES
        and "src" in path.parts
        and path.suffix == ".py"
        and path.name != "__init__.py"
    ):
        relative_modules.append(path)

relative_modules.sort(key=lambda path: path.as_posix())
corpus = [repo_root / path for path in relative_modules]

if len(corpus) < 2:
    raise RuntimeError("Self-analysis requires at least two tracked production modules.")

corpus_summary = pd.DataFrame(
    {
        "package": [path.parts[1] for path in relative_modules],
        "module": [path.as_posix() for path in relative_modules],
        "bytes": [absolute.stat().st_size for absolute in corpus],
    }
)

print(f"Repository: {repo_root}")
display(
    corpus_summary.groupby("package", sort=True)
    .agg(modules=("module", "count"), bytes=("bytes", "sum"))
    .sort_index()
)

## Pre-flight

DamiCore's cost is driven primarily by the number of objects: NCD is quadratic in object count and Neighbor Joining is cubic. `estimate()` performs the same input scan used by the run and decides whether the exact analysis fits the configured resource gates before any run artifacts are created.

In [ ]:
execution = ExecutionConfig(workers=1)
preview = estimate(
    corpus,
    source_kind="files",
    execution=execution,
)

mib = 1024**2
display(
    pd.Series(
        {
            "objects": preview.object_count,
            "pairs": preview.pair_count,
            "input MiB": preview.input_size_bytes / mib,
            "matrix MiB": preview.matrix_bytes / mib,
            "estimated working memory MiB": preview.estimated_working_memory_bytes / mib,
            "required free disk MiB": preview.required_free_disk_bytes / mib,
            "within limits": preview.within_limits,
        },
        name="pre-flight",
    ).to_frame()
)

if not preview.within_limits:
    raise RuntimeError(f"Pre-flight rejected the run: {preview.violations}")

## Run the exact pipeline

The run uses the same corpus and execution configuration that were pre-flighted. Artifacts go to a fresh temporary directory so the repository checkout is not modified.

In [ ]:
run_dir = Path(tempfile.mkdtemp(prefix="damicore-self-analysis-"))

result = run(
    corpus,
    source_kind="files",
    output_dir=run_dir,
    progress=False,
    execution=execution,
)

display(
    pd.Series(
        {
            "status": result.report.status,
            "objects": result.report.object_count,
            "clusters": result.report.cluster_count,
            "modularity": result.report.modularity,
            "run directory": str(result.artifacts.run_dir),
        },
        name="result",
    ).to_frame()
)

## Do discovered communities resemble package boundaries?

Package ownership is known independently from the clustering, so it is a useful external label for reading the result. Agreement is interesting, but disagreement is not automatically a defect: DamiCore measures compressible similarity in raw module bytes, while package boundaries encode software responsibilities.

In [ ]:
membership = result.membership.copy()
membership["package"] = membership["label"].map(lambda label: Path(label).parts[0])

display(
    membership[["cluster", "package", "label"]]
    .sort_values(["cluster", "package", "label"])
    .reset_index(drop=True)
)

display(
    pd.crosstab(
        membership["cluster"],
        membership["package"],
        rownames=["cluster"],
        colnames=["package"],
    )
)

## Which modules are closest?

Clusters summarize the global structure. The smallest off-diagonal NCD values expose the strongest pairwise similarities directly. The package-level table then compresses the same pairwise evidence into mean distances within and across packages.

In [ ]:
matrix = np.load(result.artifacts.distance_matrix, allow_pickle=False)
labels = result.membership["label"].tolist()
packages = [Path(label).parts[0] for label in labels]

pair_rows = []
for left in range(len(labels)):
    for right in range(left + 1, len(labels)):
        package_a, package_b = sorted((packages[left], packages[right]))
        pair_rows.append(
            {
                "left": labels[left],
                "right": labels[right],
                "package_a": package_a,
                "package_b": package_b,
                "ncd": float(matrix[left, right]),
            }
        )

pairs = pd.DataFrame(pair_rows).sort_values("ncd", ignore_index=True)

display(pairs[["left", "right", "ncd"]].head(15))

package_distance = pd.DataFrame(
    np.nan,
    index=sorted(PUBLIC_PACKAGES),
    columns=sorted(PUBLIC_PACKAGES),
)

for (package_a, package_b), group in pairs.groupby(["package_a", "package_b"]):
    mean_distance = group["ncd"].mean()
    package_distance.loc[package_a, package_b] = mean_distance
    package_distance.loc[package_b, package_a] = mean_distance

display(package_distance.round(3))

## Reading the result

Three observations are worth separating:

1. **Within-package concentration** suggests that modules owned by the same distribution share substantial raw-code structure.
2. **Cross-package neighbors** identify implementation similarities that cut across the architecture and are good candidates for human inspection.
3. **Cluster/package disagreement** is evidence about this representation, not a refactoring instruction.

The original DAMICORE work showed that changing the representation of code can change the structure that becomes visible. This notebook deliberately uses only raw source bytes so the experiment stays minimal and the interpretation stays clear. A future experiment could compare this baseline with a separately defined structural representation, but that is a different scientific question.

In [ ]:
result.close()
print(f"Closed the result memory map. Artifacts remain at: {run_dir}")